# Capstone Part 6: Evaluation

In this final notebook we systematically evaluate our model at each stage of the pipeline:

| Stage | Checkpoint | What it measures |
|-------|-----------|------------------|
| Base | `best_pretrained.pt` | Raw language modeling quality |
| SFT | `best_sft.pt` | Instruction-following ability |
| DPO | `best_dpo.pt` | Preference alignment |
| Quantized | `quantized_int8.pt` | Quality retention after compression |

Metrics:
1. **Perplexity** on held-out text
2. **Response quality scoring** via automated heuristics
3. **Task-specific accuracy** on simple benchmarks
4. **Comparison dashboard** showing improvement at each stage

## 1. Environment Setup

In [ ]:
!pip install -q torch datasets tiktoken matplotlib tqdm numpy

In [ ]:
import math
import time
import json
import re
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Model Architecture

In [ ]:
# ---- Model definition (same as previous notebooks) ----

@dataclass
class ModelConfig:
    vocab_size: int = 50257
    max_seq_len: int = 512
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 688
    dropout: float = 0.1
    rope_theta: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return (x * (x.float().pow(2).mean(-1, keepdim=True) + self.eps).rsqrt()).type_as(x) * self.weight


def precompute_rope_frequencies(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    angles = torch.outer(torch.arange(max_seq_len).float(), freqs)
    return torch.polar(torch.ones_like(angles), angles)


def apply_rope(x, freqs_cis):
    b, s, n, d = x.shape
    x_c = torch.view_as_complex(x.float().reshape(b, s, n, -1, 2))
    fc = freqs_cis[:s].unsqueeze(0).unsqueeze(2)
    return torch.view_as_real(x_c * fc).reshape(b, s, n, d).type_as(x)


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads, self.n_kv_heads = config.n_heads, config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.wq = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
    def _repeat_kv(self, x):
        if self.n_rep == 1: return x
        b, s, n, d = x.shape
        return x[:,:,:,None,:].expand(b,s,n,self.n_rep,d).reshape(b,s,self.n_heads,d)
    def forward(self, x, freqs_cis, mask=None):
        b, s, _ = x.shape
        q = self.wq(x).reshape(b,s,self.n_heads,self.head_dim)
        k = self.wk(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        v = self.wv(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        q, k = apply_rope(q, freqs_cis), apply_rope(k, freqs_cis)
        k, v = self._repeat_kv(k), self._repeat_kv(v)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        attn = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        if mask is not None: attn = attn.masked_fill(mask == 0, float("-inf"))
        out = torch.matmul(self.attn_dropout(F.softmax(attn, dim=-1)), v)
        return self.resid_dropout(self.wo(out.transpose(1,2).reshape(b,s,-1)))


class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w2 = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.w3 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_norm = RMSNorm(config.d_model)
        self.attention = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config.d_model)
        self.ffn = SwiGLU(config)
    def forward(self, x, freqs_cis, mask=None):
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        return x + self.ffn(self.ffn_norm(x))


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        head_dim = config.d_model // config.n_heads
        self.register_buffer("freqs_cis", precompute_rope_frequencies(head_dim, config.max_seq_len, config.rope_theta), persistent=False)
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, input_ids, targets=None):
        b, s = input_ids.shape
        mask = torch.tril(torch.ones(s, s, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        x = self.dropout(self.tok_emb(input_ids))
        for layer in self.layers:
            x = layer(x, self.freqs_cis, mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=100, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            input_ids = torch.cat([input_ids, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
        return input_ids


print("Model architecture defined.")

## 3. Load All Checkpoints

In [ ]:
save_dir = Path("checkpoints")
tokenizer = tiktoken.get_encoding("gpt2")

INSTRUCTION_PREFIX = "<|instruction|> "
RESPONSE_PREFIX = " <|response|> "
END_TOKEN = "<|end|>"

# Load all stage checkpoints
stages = {}

checkpoint_files = {
    "Base (Pretrained)": "best_pretrained.pt",
    "SFT": "best_sft.pt",
    "DPO": "best_dpo.pt",
}

for stage_name, filename in checkpoint_files.items():
    ckpt_path = save_dir / filename
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        config = ckpt["config"]
        m = MiniLLM(config).to(device)
        m.load_state_dict(ckpt["model_state_dict"])
        m.eval()
        stages[stage_name] = m
        print(f"Loaded {stage_name} from {filename}")
    else:
        print(f"WARNING: {filename} not found, skipping {stage_name}")

# Load quantized model (INT8 dynamic)
int8_path = save_dir / "quantized_int8.pt"
if int8_path.exists():
    int8_ckpt = torch.load(int8_path, map_location="cpu", weights_only=False)
    int8_config = int8_ckpt["config"]
    int8_model = MiniLLM(int8_config)
    # Apply dynamic quantization
    int8_model = torch.ao.quantization.quantize_dynamic(
        int8_model, {nn.Linear}, dtype=torch.qint8
    )
    int8_model.eval()
    # Note: We load the DPO model and re-quantize for a fair comparison
    if "DPO" in stages:
        dpo_cpu = MiniLLM(config)
        dpo_ckpt = torch.load(save_dir / "best_dpo.pt", map_location="cpu", weights_only=False)
        dpo_cpu.load_state_dict(dpo_ckpt["model_state_dict"])
        dpo_cpu.eval()
        int8_model = torch.ao.quantization.quantize_dynamic(
            dpo_cpu, {nn.Linear}, dtype=torch.qint8
        )
        int8_model.eval()
        stages["Quantized (INT8)"] = int8_model
        print("Loaded Quantized (INT8) model")
    del int8_ckpt

print(f"\nTotal models loaded: {len(stages)}")

## 4. Evaluation Dataset

In [ ]:
from datasets import load_dataset

# Load eval set for perplexity
raw_eval = load_dataset("roneneldan/TinyStories", split="validation[:500]")

eval_tokens = []
for ex in raw_eval:
    tokens = tokenizer.encode(ex["text"])
    eval_tokens.extend(tokens)

seq_len = config.max_seq_len
eval_tensor = torch.tensor(eval_tokens[:len(eval_tokens) // (seq_len + 1) * (seq_len + 1)])
eval_chunks = eval_tensor.reshape(-1, seq_len + 1)
eval_inputs = eval_chunks[:, :-1]
eval_targets = eval_chunks[:, 1:]

print(f"Eval set: {eval_chunks.shape[0]} chunks, {eval_chunks.numel():,} total tokens")

## 5. Metric 1: Perplexity

Perplexity measures how well the model predicts the next token on held-out data.
Lower is better. It is the exponential of the average cross-entropy loss.

In [ ]:
@torch.no_grad()
def compute_perplexity(model, eval_inputs, eval_targets, batch_size=16):
    """Compute perplexity on eval set."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    model_device = next(model.parameters()).device

    for i in range(0, len(eval_inputs), batch_size):
        x = eval_inputs[i:i + batch_size].to(model_device)
        y = eval_targets[i:i + batch_size].to(model_device)
        logits, _ = model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            reduction='sum'
        )
        total_loss += loss.item()
        total_tokens += y.numel()

    return math.exp(total_loss / total_tokens)


print("Computing perplexity for each stage...")
perplexity_results = {}
for stage_name, model in stages.items():
    ppl = compute_perplexity(model, eval_inputs, eval_targets)
    perplexity_results[stage_name] = ppl
    print(f"  {stage_name}: {ppl:.2f}")

## 6. Metric 2: Response Quality Scoring

We use automated heuristics to score response quality:
- **Coherence**: Does the response stay on topic? (keyword overlap with prompt)
- **Completeness**: Does it form complete sentences?
- **Verbosity**: Is the response sufficiently detailed? (word count)
- **Repetition**: Does it avoid excessive repetition?
- **Fluency**: Is word diversity reasonable?

In [ ]:
def score_response(instruction: str, response: str) -> dict:
    """Score a response on multiple quality dimensions (0-1 each)."""
    scores = {}
    words = response.split()
    sentences = [s.strip() for s in re.split(r'[.!?]', response) if s.strip()]

    # 1. Coherence: keyword overlap between instruction and response
    instr_words = set(instruction.lower().split())
    stop_words = {'a', 'the', 'is', 'was', 'to', 'and', 'of', 'in', 'on', 'it', 'for', 'that', 'with'}
    instr_keywords = instr_words - stop_words
    if instr_keywords:
        resp_words_lower = set(w.lower().strip('.,!?') for w in words)
        overlap = len(instr_keywords & resp_words_lower) / len(instr_keywords)
        scores['coherence'] = min(overlap * 2, 1.0)  # Scale up, cap at 1
    else:
        scores['coherence'] = 0.5

    # 2. Completeness: ends with punctuation and has multiple sentences
    ends_with_punct = response.rstrip().endswith(('.', '!', '?', '"'))
    has_multiple_sentences = len(sentences) >= 2
    scores['completeness'] = (0.5 * ends_with_punct + 0.5 * has_multiple_sentences)

    # 3. Verbosity: word count (20-100 is ideal for our model)
    n_words = len(words)
    if n_words < 5:
        scores['verbosity'] = 0.1
    elif n_words < 20:
        scores['verbosity'] = 0.4
    elif n_words <= 120:
        scores['verbosity'] = 1.0
    else:
        scores['verbosity'] = max(0.5, 1.0 - (n_words - 120) / 200)

    # 4. Repetition: ratio of unique n-grams to total n-grams
    if len(words) >= 4:
        trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
        unique_ratio = len(set(trigrams)) / len(trigrams) if trigrams else 1.0
        scores['non_repetition'] = unique_ratio
    else:
        scores['non_repetition'] = 0.5

    # 5. Fluency: type-token ratio (vocabulary diversity)
    if words:
        unique_words = set(w.lower().strip('.,!?') for w in words)
        ttr = len(unique_words) / len(words)
        scores['fluency'] = min(ttr * 2, 1.0)  # Good TTR is ~0.5+
    else:
        scores['fluency'] = 0.0

    # Overall score (weighted average)
    weights = {
        'coherence': 0.25,
        'completeness': 0.20,
        'verbosity': 0.20,
        'non_repetition': 0.20,
        'fluency': 0.15,
    }
    scores['overall'] = sum(scores[k] * weights[k] for k in weights)

    return scores


# Test the scorer
test_scores = score_response(
    "Write a story about a cat.",
    "Once upon a time, there was a little cat named Whiskers. She loved to play in the garden. One day, she found a butterfly and chased it around. She was happy."
)
print("Test scoring:")
for k, v in test_scores.items():
    print(f"  {k}: {v:.3f}")

In [ ]:
# Evaluation instructions
eval_instructions = [
    "Write a short story about a cat.",
    "Tell a story about friendship.",
    "Write about a sunny day.",
    "Tell a bedtime story.",
    "Write a story about the ocean.",
    "Tell a story about being brave.",
    "Write about a birthday party.",
    "Tell a story about a dog.",
    "Write about learning something new.",
    "Tell a story about sharing.",
]


def generate_response(model, instruction, max_tokens=100):
    """Generate a response for evaluation."""
    prompt = INSTRUCTION_PREFIX + instruction + RESPONSE_PREFIX
    model_device = next(model.parameters()).device
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=model_device)
    output_ids = model.generate(input_ids, max_new_tokens=max_tokens, temperature=0.7, top_k=40)
    full_text = tokenizer.decode(output_ids[0].tolist())
    if "<|response|>" in full_text:
        resp = full_text.split("<|response|>")[-1]
        if END_TOKEN in resp:
            resp = resp.split(END_TOKEN)[0]
        return resp.strip()
    return full_text[len(prompt):].strip()


# Generate responses from each stage
print("Generating responses from each stage...")
all_responses = {}  # {stage: {instruction: response}}
all_scores = {}     # {stage: {instruction: scores_dict}}

for stage_name, model in tqdm(stages.items(), desc="Stages"):
    all_responses[stage_name] = {}
    all_scores[stage_name] = {}
    for instr in eval_instructions:
        resp = generate_response(model, instr)
        scores = score_response(instr, resp)
        all_responses[stage_name][instr] = resp
        all_scores[stage_name][instr] = scores

print("Done generating and scoring responses.")

In [ ]:
# Compute average scores per stage
avg_scores = {}
for stage_name in stages:
    stage_scores = defaultdict(list)
    for instr in eval_instructions:
        for metric, val in all_scores[stage_name][instr].items():
            stage_scores[metric].append(val)
    avg_scores[stage_name] = {k: np.mean(v) for k, v in stage_scores.items()}

# Display
print("\nAverage Quality Scores by Stage:")
print(f"{'Stage':<22} {'Coherence':>10} {'Complete':>10} {'Verbose':>10} {'NonRepeat':>10} {'Fluency':>10} {'Overall':>10}")
print("-" * 85)
for stage_name in stages:
    s = avg_scores[stage_name]
    print(
        f"{stage_name:<22} "
        f"{s['coherence']:>10.3f} "
        f"{s['completeness']:>10.3f} "
        f"{s['verbosity']:>10.3f} "
        f"{s['non_repetition']:>10.3f} "
        f"{s['fluency']:>10.3f} "
        f"{s['overall']:>10.3f}"
    )

## 7. Metric 3: Task-Specific Accuracy

Simple benchmarks to test specific capabilities:
- **Story completion**: Does the model complete a story coherently?
- **Instruction following**: Does the response match the requested format?
- **Factual consistency**: Within a story, are details consistent?

In [ ]:
# Task 1: Story completion (does the model generate a story when asked?)
story_keywords = ['once', 'upon', 'time', 'day', 'went', 'said', 'was', 'had', 'then',
                   'little', 'big', 'happy', 'friend', 'play', 'love', 'found', 'came']

def check_story_quality(response):
    """Check if a response looks like a story."""
    words = response.lower().split()
    if len(words) < 10:
        return 0.0
    keyword_hits = sum(1 for kw in story_keywords if kw in words)
    has_narrative = any(w in words for w in ['then', 'after', 'next', 'finally', 'suddenly'])
    has_characters = any(w[0].isupper() for w in response.split() if len(w) > 1)
    score = min(keyword_hits / 5, 1.0) * 0.5 + has_narrative * 0.25 + has_characters * 0.25
    return score


# Task 2: Instruction format adherence
def check_format_adherence(instruction, response):
    """Check if the response follows the instruction format."""
    # Check if it's not just repeating the instruction
    instr_words = set(instruction.lower().split())
    resp_words = set(response.lower().split())
    if len(resp_words) < 5:
        return 0.0
    # Response should be longer than instruction
    length_ok = len(response.split()) > len(instruction.split())
    # Response should not be identical to instruction
    not_parrot = response.strip().lower() != instruction.strip().lower()
    # Response has complete sentences
    has_sentences = response.rstrip().endswith(('.', '!', '?'))
    return (length_ok * 0.4 + not_parrot * 0.3 + has_sentences * 0.3)


print("Computing task-specific accuracy...")
task_results = {}

for stage_name in stages:
    story_scores = []
    format_scores = []

    for instr in eval_instructions:
        resp = all_responses[stage_name][instr]
        story_scores.append(check_story_quality(resp))
        format_scores.append(check_format_adherence(instr, resp))

    task_results[stage_name] = {
        "story_quality": np.mean(story_scores),
        "format_adherence": np.mean(format_scores),
    }
    print(f"  {stage_name}:")
    print(f"    Story quality: {np.mean(story_scores):.3f}")
    print(f"    Format adherence: {np.mean(format_scores):.3f}")

## 8. Sample Responses Side-by-Side

In [ ]:
# Show a few examples
sample_instructions = eval_instructions[:3]

for instr in sample_instructions:
    print("=" * 80)
    print(f"INSTRUCTION: {instr}")
    print("=" * 80)
    for stage_name in stages:
        resp = all_responses[stage_name][instr]
        overall = all_scores[stage_name][instr]['overall']
        print(f"\n  [{stage_name}] (score: {overall:.3f})")
        print(f"  {resp[:200]}")
    print()

## 9. Comparison Dashboard

The main visualization: a comprehensive dashboard showing improvement across all stages.

In [ ]:
fig = plt.figure(figsize=(18, 14))

stage_names = list(stages.keys())
n_stages = len(stage_names)
colors = plt.cm.Set2(np.linspace(0, 1, n_stages))

# 1. Perplexity (top-left)
ax1 = fig.add_subplot(2, 3, 1)
ppls = [perplexity_results.get(s, 0) for s in stage_names]
bars = ax1.bar(range(n_stages), ppls, color=colors)
ax1.set_xticks(range(n_stages))
ax1.set_xticklabels(stage_names, rotation=30, ha='right', fontsize=9)
ax1.set_ylabel('Perplexity')
ax1.set_title('Perplexity (Lower = Better)')
for bar, val in zip(bars, ppls):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# 2. Overall quality score (top-center)
ax2 = fig.add_subplot(2, 3, 2)
overall_scores = [avg_scores[s]['overall'] for s in stage_names]
bars = ax2.bar(range(n_stages), overall_scores, color=colors)
ax2.set_xticks(range(n_stages))
ax2.set_xticklabels(stage_names, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('Score (0-1)')
ax2.set_title('Overall Response Quality')
ax2.set_ylim(0, 1)
for bar, val in zip(bars, overall_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Radar chart of quality dimensions (top-right)
ax3 = fig.add_subplot(2, 3, 3, polar=True)
dimensions = ['coherence', 'completeness', 'verbosity', 'non_repetition', 'fluency']
dim_labels = ['Coherence', 'Complete', 'Verbosity', 'Non-Repeat', 'Fluency']
angles = np.linspace(0, 2 * np.pi, len(dimensions), endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

for i, stage_name in enumerate(stage_names):
    values = [avg_scores[stage_name][d] for d in dimensions]
    values += values[:1]
    ax3.plot(angles, values, 'o-', linewidth=2, label=stage_name, color=colors[i])
    ax3.fill(angles, values, alpha=0.1, color=colors[i])

ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(dim_labels, fontsize=8)
ax3.set_ylim(0, 1)
ax3.set_title('Quality Dimensions', fontsize=11, pad=20)
ax3.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

# 4. Per-dimension breakdown (bottom-left)
ax4 = fig.add_subplot(2, 3, 4)
x = np.arange(len(dimensions))
width = 0.8 / n_stages
for i, stage_name in enumerate(stage_names):
    vals = [avg_scores[stage_name][d] for d in dimensions]
    offset = (i - n_stages/2 + 0.5) * width
    ax4.bar(x + offset, vals, width, label=stage_name, color=colors[i])

ax4.set_xticks(x)
ax4.set_xticklabels(dim_labels, rotation=30, ha='right', fontsize=9)
ax4.set_ylabel('Score (0-1)')
ax4.set_title('Quality Dimensions Breakdown')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3, axis='y')
ax4.set_ylim(0, 1.1)

# 5. Task accuracy (bottom-center)
ax5 = fig.add_subplot(2, 3, 5)
task_dims = ['story_quality', 'format_adherence']
task_labels = ['Story Quality', 'Format Adherence']
x = np.arange(len(task_dims))
width = 0.8 / n_stages
for i, stage_name in enumerate(stage_names):
    vals = [task_results[stage_name][d] for d in task_dims]
    offset = (i - n_stages/2 + 0.5) * width
    ax5.bar(x + offset, vals, width, label=stage_name, color=colors[i])

ax5.set_xticks(x)
ax5.set_xticklabels(task_labels, fontsize=10)
ax5.set_ylabel('Score (0-1)')
ax5.set_title('Task-Specific Accuracy')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3, axis='y')
ax5.set_ylim(0, 1.1)

# 6. Improvement waterfall (bottom-right)
ax6 = fig.add_subplot(2, 3, 6)
if len(stage_names) > 1:
    improvements = []
    labels = []
    base_score = overall_scores[0]
    for i in range(1, len(stage_names)):
        delta = overall_scores[i] - overall_scores[i-1]
        improvements.append(delta)
        labels.append(f"{stage_names[i-1]}\n-> {stage_names[i]}")

    bar_colors = ['green' if v > 0 else 'red' for v in improvements]
    ax6.bar(range(len(improvements)), improvements, color=bar_colors, alpha=0.7)
    ax6.set_xticks(range(len(improvements)))
    ax6.set_xticklabels(labels, fontsize=8, ha='center')
    ax6.set_ylabel('Quality Change')
    ax6.set_title('Stage-over-Stage Improvement')
    ax6.axhline(y=0, color='gray', linestyle='--')
    ax6.grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(improvements):
        ax6.text(i, v, f'{v:+.3f}', ha='center',
                 va='bottom' if v > 0 else 'top', fontweight='bold', fontsize=9)

plt.suptitle('Model Evaluation Dashboard: Build Your Own LLM',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(save_dir / 'evaluation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Response Length Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for i, stage_name in enumerate(stage_names):
    lengths = [len(all_responses[stage_name][instr].split()) for instr in eval_instructions]
    positions = np.array(range(len(eval_instructions))) + i * 0.15 - 0.15 * n_stages / 2
    ax.scatter(positions, lengths, s=60, color=colors[i], label=stage_name, alpha=0.7, zorder=3)

ax.set_xticks(range(len(eval_instructions)))
ax.set_xticklabels([f"Q{i+1}" for i in range(len(eval_instructions))])
ax.set_ylabel('Response Length (words)')
ax.set_title('Response Length Distribution by Stage')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Stats
print("Average response length (words):")
for stage_name in stage_names:
    lengths = [len(all_responses[stage_name][instr].split()) for instr in eval_instructions]
    print(f"  {stage_name}: {np.mean(lengths):.1f} (std: {np.std(lengths):.1f})")

## 11. Per-Instruction Score Heatmap

In [ ]:
# Create a heatmap of overall scores: rows=instructions, columns=stages
score_matrix = np.zeros((len(eval_instructions), n_stages))
for j, stage_name in enumerate(stage_names):
    for i, instr in enumerate(eval_instructions):
        score_matrix[i, j] = all_scores[stage_name][instr]['overall']

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(score_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(n_stages))
ax.set_xticklabels(stage_names, rotation=30, ha='right')
ax.set_yticks(range(len(eval_instructions)))
ax.set_yticklabels([f"Q{i+1}: {instr[:30]}..." for i, instr in enumerate(eval_instructions)],
                    fontsize=9)

# Add text annotations
for i in range(len(eval_instructions)):
    for j in range(n_stages):
        text = ax.text(j, i, f"{score_matrix[i, j]:.2f}",
                       ha="center", va="center", fontsize=9,
                       color="white" if score_matrix[i, j] > 0.6 else "black")

plt.colorbar(im, ax=ax, label='Overall Quality Score')
ax.set_title('Quality Score Heatmap: Instruction x Stage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Final Summary Report

In [ ]:
print("=" * 80)
print("FINAL EVALUATION REPORT: Build Your Own LLM")
print("=" * 80)

print("\n--- Model Architecture ---")
print(f"Parameters: ~{sum(p.numel() for p in list(stages.values())[0].parameters()) / 1e6:.1f}M")
print(f"Architecture: RoPE + RMSNorm + SwiGLU + GQA")
print(f"Vocabulary: 50,257 (GPT-2 tokenizer)")
print(f"Context length: {config.max_seq_len}")

print("\n--- Pipeline Stages ---")
print(f"{'Stage':<22} {'Perplexity':>12} {'Quality':>10} {'Story':>10} {'Format':>10}")
print("-" * 67)
for stage_name in stage_names:
    ppl = perplexity_results.get(stage_name, float('nan'))
    quality = avg_scores[stage_name]['overall']
    story = task_results[stage_name]['story_quality']
    fmt = task_results[stage_name]['format_adherence']
    print(f"{stage_name:<22} {ppl:>12.2f} {quality:>10.3f} {story:>10.3f} {fmt:>10.3f}")

print("\n--- Key Findings ---")
if len(stage_names) >= 2:
    best_stage = max(stage_names, key=lambda s: avg_scores[s]['overall'])
    worst_stage = min(stage_names, key=lambda s: avg_scores[s]['overall'])
    improvement = avg_scores[best_stage]['overall'] - avg_scores[worst_stage]['overall']
    print(f"Best stage: {best_stage} (overall: {avg_scores[best_stage]['overall']:.3f})")
    print(f"Total quality improvement: {improvement:+.3f} ({improvement/avg_scores[worst_stage]['overall']*100:+.1f}%)")

best_ppl = min(stage_names, key=lambda s: perplexity_results.get(s, float('inf')))
print(f"Best perplexity: {best_ppl} ({perplexity_results[best_ppl]:.2f})")

print("\n--- Conclusions ---")
print("1. Pretraining establishes language modeling ability (coherent text generation)")
print("2. SFT teaches instruction-following format and improves response relevance")
print("3. DPO aligns preferences toward more detailed, engaging responses")
print("4. INT8 quantization preserves most quality while halving model size")
print("\nThe full pretrain -> SFT -> DPO -> quantize pipeline demonstrates")
print("the modern LLM development workflow at educational scale.")

In [ ]:
# Save evaluation results to JSON for reference
eval_report = {
    "perplexity": {s: float(v) for s, v in perplexity_results.items()},
    "quality_scores": {s: {k: float(v) for k, v in scores.items()} for s, scores in avg_scores.items()},
    "task_accuracy": {s: {k: float(v) for k, v in scores.items()} for s, scores in task_results.items()},
}

with open(save_dir / "evaluation_report.json", "w") as f:
    json.dump(eval_report, f, indent=2)

print(f"\nEvaluation report saved to {save_dir / 'evaluation_report.json'}")
print(f"Dashboard figure saved to {save_dir / 'evaluation_dashboard.png'}")

## Summary

In this final notebook we:

1. **Loaded** all checkpoints (pretrained, SFT, DPO, quantized)
2. **Computed** perplexity on held-out TinyStories data
3. **Scored** response quality with automated heuristics (coherence, completeness, etc.)
4. **Evaluated** task-specific accuracy (story quality, format adherence)
5. **Built** a comprehensive comparison dashboard
6. **Generated** a final evaluation report

This concludes the Build Your Own LLM capstone project. You have successfully:
- Pretrained a modern transformer from scratch
- Fine-tuned it on instructions
- Aligned it with DPO
- Quantized it for efficient serving
- Deployed it with a production-style API
- Evaluated it systematically at every stage

**Congratulations on completing Section 01!**